# 🛣️ Smart City Road-Defect & Pothole Detection — YOLOv8 Training Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokitheeditor697-create/Pathole/blob/main/train_pothole_yolov8.ipynb)

This notebook trains a custom **YOLOv8** model for real-time onboard edge detection of road potholes and defects on transit buses.

### Step 1: Check GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow opencv-python matplotlib

### Step 2: Download Public Pothole Dataset

In [ ]:
# Download benchmark road pothole dataset (Roboflow Universe / Public)
!curl -L "https://universe.roboflow.com/ds/t6yR1o0mR0?key=3sL2Zl6yGz" > roboflow.zip; unzip -q roboflow.zip -d ./pothole_dataset; rm roboflow.zip

### Step 3: Train YOLOv8 Model on GPU

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 Nano model (optimized for Raspberry Pi / Jetson edge hardware)
model = YOLO('yolov8n.pt')

# Train for 50 epochs
results = model.train(
    data='./pothole_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    name='pothole_yolov8_model'
)

### Step 4: Evaluate Model Performance (mAP, Loss & Confusion Matrix)

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Validate the model
metrics = model.val()
print(f"Validation mAP50: {metrics.box.map50:.4f}")
print(f"Validation mAP50-95: {metrics.box.map:.4f}")

# Display training loss curves & confusion matrix
results_img = cv2.imread('runs/detect/pothole_yolov8_model/results.png')
if results_img is not None:
    plt.figure(figsize=(16, 10))
    plt.imshow(cv2.cvtColor(results_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Training Loss & Validation Metrics')
    plt.show()

### Step 5: Test Model on Test Images

In [ ]:
# Run prediction on test set
preds = model.predict(source='./pothole_dataset/test/images', conf=0.5, save=True)
print("Predictions saved to runs/detect/predict/")

### Step 6: Download the Trained Model (`best.pt`)

In [ ]:
from google.colab import files
files.download('runs/detect/pothole_yolov8_model/weights/best.pt')